# judgeman: fine-tune a small judge (Leg 2)

Trains a LoRA adapter on Qwen 3.5 so it judges agent steps better, then has it answer the exam prompts.

**Settings (right panel → Session options):** Accelerator **GPU T4 x2**, Internet **On**, and the dataset with `train.jsonl`, `val.jsonl`, `exam.jsonl` attached as an input.

**Two runs, each started with *Save Version → Save & Run All (Commit)* so they keep going with the tab closed:**
1. `MODE = "train"`: fine-tunes, then answers the exam. About 7 to 8 hours. Output: `exam-replies.jsonl` and the `adapter` folder.
2. `MODE = "base"`: answers the exam with the untrained model, for a like-for-like baseline. About 2 hours. Output: `exam-replies-base.jsonl`.

`TRIAL = True` shrinks either run to a few minutes, to check the notebook works.

In [ ]:
%%capture
!pip install --upgrade --no-cache-dir unsloth unsloth_zoo

In [ ]:
MODE = "train"          # "train": fine-tune, then answer the exam.  "base": answer the exam untrained.
TRIAL = False           # True: tiny run to check the notebook works.
MODEL = "unsloth/Qwen3.5-9B"
MAX_SEQ = 4096          # longest training example, in tokens
EPOCHS = 1              # measured on a T4: about 18 s per example, so one pass is about 5.5 hours
BATCH = 4               # exam prompts answered at once

import glob, json, os, time
DATA = os.path.dirname(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0])
OUT = "/kaggle/working"
read = lambda name: [json.loads(l) for l in open(f"{DATA}/{name}.jsonl")]
train_rows, val_rows, exam_rows = read("train"), read("val"), read("exam")
if TRIAL:
    train_rows, val_rows, exam_rows = train_rows[:40], val_rows[:10], exam_rows[:30]
print(MODE, "|", len(train_rows), "train,", len(val_rows), "val,", len(exam_rows), "exam prompts from", DATA)

In [ ]:
import torch
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, full_finetuning=False,
)
# Qwen 3.5 is a vision-language model, so `tokenizer` is really a processor whose first
# argument is an image. `tok` is the plain text tokenizer inside it.
tok = getattr(tokenizer, "tokenizer", tokenizer)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
print(torch.cuda.get_device_name(0), f"{torch.cuda.memory_allocated() / 2**30:.1f} GB used after loading")

In [ ]:
def render(messages, for_answer):
    # thinking is off: the judge prompt already asks for a short critique before the verdict
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=for_answer, enable_thinking=False
    )

def answer_exam(rows, path):
    """Answer exam prompts in batches and append to `path`. Finished prompts are skipped on re-run."""
    done = {(r["trajectory_id"], r["step"]) for r in map(json.loads, open(path))} if os.path.exists(path) else set()
    todo = [r for r in rows if (r["trajectory_id"], r["step"]) not in done]
    todo.sort(key=lambda r: len(r["messages"][1]["content"]))  # similar lengths together, less padding
    tok.padding_side = "left"  # the answer continues from the right-hand end of every prompt
    FastLanguageModel.for_inference(model)
    started = time.time()
    with open(path, "a") as f:
        for i in range(0, len(todo), BATCH):
            batch = todo[i : i + BATCH]
            inputs = tok([render(r["messages"], True) for r in batch], return_tensors="pt",
                         padding=True).to(model.device)
            t0 = time.time()
            with torch.no_grad():
                out = model.generate(**inputs, max_new_tokens=180, do_sample=False)
            each = round((time.time() - t0) / len(batch), 2)
            for row, tokens in zip(batch, out):
                reply = tok.decode(tokens[inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
                f.write(json.dumps({"trajectory_id": row["trajectory_id"], "step": row["step"],
                                    "reply": reply, "seconds": each}) + "\n")
            f.flush()
            if (i // BATCH) % 10 == 0:
                print(i, "of", len(todo), f"{(time.time() - started) / 60:.0f} min", reply[:80].replace("\n", " "))
    tok.padding_side = "right"

# a two-batch check first, so a problem with answering shows up now and not after hours of training
answer_exam(exam_rows[:2 * BATCH], f"{OUT}/smoke-check.jsonl")
if MODE == "base":
    answer_exam(exam_rows, f"{OUT}/exam-replies-base.jsonl")
    print("Done. Download exam-replies-base.jsonl from the Output tab.")

In [ ]:
if MODE == "train":
    from datasets import Dataset
    from trl import SFTConfig, SFTTrainer

    FastLanguageModel.for_training(model)
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=16, lora_dropout=0, bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        use_gradient_checkpointing="unsloth", random_state=3407, max_seq_length=MAX_SEQ,
    )
    to_text = lambda rows: Dataset.from_list([{"text": render(r["messages"], False)} for r in rows])
    trainer = SFTTrainer(
        model=model, tokenizer=tok, train_dataset=to_text(train_rows), eval_dataset=to_text(val_rows),
        args=SFTConfig(
            dataset_text_field="text", max_length=MAX_SEQ,
            per_device_train_batch_size=1, gradient_accumulation_steps=8,
            num_train_epochs=EPOCHS, max_steps=5 if TRIAL else -1,
            learning_rate=2e-4, warmup_ratio=0.05, lr_scheduler_type="cosine", weight_decay=0.01,
            logging_steps=1 if TRIAL else 5, eval_strategy="no", save_strategy="no",
            optim="adamw_8bit", seed=3407, output_dir=f"{OUT}/checkpoints", report_to="none",
        ),
    )
    # Learn from the answer only. Without this, most of the training signal goes into
    # predicting the long prompt (logs and code) and little into the 80-token verdict.
    try:
        from unsloth.chat_templates import train_on_responses_only
        trainer = train_on_responses_only(
            trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n"
        )
        print("training on the answers only")
    except Exception as e:
        print("could not mask the prompts, training on the full text instead:", repr(e)[:300])

    stats = trainer.train()
    print(stats.metrics, f"peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
    model.save_pretrained(f"{OUT}/adapter"); tok.save_pretrained(f"{OUT}/adapter")

In [ ]:
if MODE == "train":
    answer_exam(exam_rows, f"{OUT}/exam-replies.jsonl")
    print("Done. Download exam-replies.jsonl from the Output tab.")